In [41]:
import pandas as pd

from google.cloud import bigquery
from google.oauth2 import service_account

# Environment
from dotenv import dotenv_values
import os
from dotenv import load_dotenv

In [42]:
for year in range(2021, 2025):
    exports = pd.read_csv(
        f"../data/raw/Exports/EXP_{year}.csv",
        sep=";",
        encoding="latin1"
    )
    exports.to_csv(
        f"../data/processed/Exports/EXP_{year}.csv",
        index=False,
        sep=",",         
        encoding="utf-8"  # UTF-8
    )

In [43]:
for year in range(2021, 2025):

    imports = pd.read_csv(
        f"../data/raw/Imports/IMP_{year}.csv",
        sep=";",
        encoding="latin1"
    )

    imports.to_csv(
        f"../data/processed/Imports/IMP_{year}.csv",
        index=False,
        sep=",",
        encoding="utf-8"
    )

In [44]:
ncm_cgce = pd.read_csv("../data/raw/NCM_CGCE.csv", 
                       sep = ";", 
                       encoding="latin1")
ncm_cgce.to_csv("../data/processed/Dimension/NCM_CGCE.csv",
                index=False,
                sep=",",
                encoding="utf-8")


In [45]:
ncm_cuci = pd.read_csv("../data/raw/NCM_CUCI.csv", 
                       sep = ";", 
                       encoding="latin1")
ncm_cuci.to_csv("../data/processed/Dimension/NCM_CUCI.csv",
                index=False,
                sep=",",
                encoding="utf-8")

All raw tables were initially loaded into BigQuery via the Google Cloud Console. Due to schema detection issues during manual import, the PAIS and NCM tables were instead loaded programmatically using Python and the BigQuery client library.

In [46]:
ncm = pd.read_csv("../data/raw/NCM.csv", 
                       sep = ";", 
                       encoding="latin1")
ncm.to_csv("../data/processed/Dimension/NCM.csv",
                index=False,
                sep=",",
                encoding="utf-8")

In [47]:
pais = pd.read_csv("../data/raw/PAIS.csv", 
                       sep = ";", 
                       encoding="latin1")
pais.to_csv("../data/processed/Dimension/PAIS.csv",
                index=False,
                sep=",",
                encoding="utf-8")

In [48]:
pais.head()

,CO_PAIS,CO_PAIS_ISON3,CO_PAIS_ISOA3,NO_PAIS,NO_PAIS_ING,NO_PAIS_ESP
0,0,898,ZZZ,Não Definido,Not defined,No definido
1,13,4,AFG,Afeganistão,Afghanistan,Afganistan
2,15,248,ALA,"Aland, Ilhas",Aland Islands,"Alans, Islas"
3,17,8,ALB,Albânia,Albania,Albania
4,20,724,ESP,"Alboran-Perejil, Ilhas","Alboran-Perejil, Islands","Alboran-Perejil, Islas"


In [49]:
pais.isnull().sum()

CO_PAIS          0
CO_PAIS_ISON3    0
CO_PAIS_ISOA3    0
NO_PAIS          0
NO_PAIS_ING      0
NO_PAIS_ESP      0
dtype: int64

In [50]:
pais.columns

Index(['CO_PAIS', 'CO_PAIS_ISON3', 'CO_PAIS_ISOA3', 'NO_PAIS', 'NO_PAIS_ING',
       'NO_PAIS_ESP'],
      dtype='object')

In [51]:
load_dotenv()

credentials = service_account.Credentials.from_service_account_file(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)

client = bigquery.Client(
    credentials=credentials,
    project=os.getenv("PROJECT_ID")
)

In [52]:
# Connection to BigQuery
client = bigquery.Client(project="foreign-trade-brazil")

# Read processed table
df_pais = pd.read_csv("../data/processed/Dimension/PAIS.csv")

# Destination Table
table_id = "foreign-trade-brazil.Dimension.PAIS"

# Configurations
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Uploaded to BigQuery
job = client.load_table_from_dataframe(
    df_pais,
    table_id,
    job_config=job_config
)

job.result()

print(f"{len(df_pais)} records sent to {table_id}.")

281 records sent to foreign-trade-brazil.Dimension.PAIS.


In [53]:
ncm.head()

,CO_NCM,CO_UNID,CO_SH6,CO_PPE,CO_PPI,CO_FAT_AGREG,CO_CUCI_ITEM,CO_CGCE_N3,CO_SIIT,CO_ISIC_CLASSE,CO_EXP_SUBSET,NO_NCM_POR,NO_NCM_ESP,NO_NCM_ING
0,49011000,10,490110,3521.0,3521.0,3.0,89215,322,9000.0,5811,8099.0,"Livros, brochuras e impressos semelhantes, em ...","Libros, folletos, impresos similares, en hojas...","Books, brochures, similar printed papers, in s..."
1,49019100,10,490191,3521.0,3521.0,3.0,89216,322,9000.0,5811,8099.0,"Dicionários e enciclopédias, mesmo em fascículos","Diccionarios y enciclopedias, incluso en fascí...","Dictionaries and encyclopaedias, serial instal..."
2,49019900,10,490199,3521.0,3521.0,3.0,89219,322,9000.0,5811,8099.0,"Outros livros, brochuras e impressos semelhantes","Otros libros, folletos y impresos similares","Oth.printed books, brochures and similar printed"
3,49021000,10,490210,3521.0,3521.0,3.0,89221,322,9000.0,5813,8099.0,"Jornais e publicações periódicas, impressos, m...","Diarios y publicaciones, impresos, periodo>=4 ...","Newspapers, journals, printed, periodical>=4 t..."
4,52102910,10,521029,3840.0,3840.0,3.0,65251,240,4000.0,1312,1605.0,Tecidos de algodão que contenham menos de 85 %...,"Tejidos de algodón, blanqueados, con fibras si...","Fabrics of cotton, bleached, synthetic or arti..."


In [54]:
ncm.isnull().sum()

CO_NCM              0
CO_UNID             0
CO_SH6              0
CO_PPE            298
CO_PPI            298
CO_FAT_AGREG       20
CO_CUCI_ITEM        0
CO_CGCE_N3          0
CO_SIIT           298
CO_ISIC_CLASSE      0
CO_EXP_SUBSET     299
NO_NCM_POR          0
NO_NCM_ESP          0
NO_NCM_ING          0
dtype: int64

In [55]:
ncm.columns

Index(['CO_NCM', 'CO_UNID', 'CO_SH6', 'CO_PPE', 'CO_PPI', 'CO_FAT_AGREG',
       'CO_CUCI_ITEM', 'CO_CGCE_N3', 'CO_SIIT', 'CO_ISIC_CLASSE',
       'CO_EXP_SUBSET', 'NO_NCM_POR', 'NO_NCM_ESP', 'NO_NCM_ING'],
      dtype='object')

In [56]:
# Connection with BigQuery
client = bigquery.Client(project="foreign-trade-brazil")

# Read the processed file
df_ncm = pd.read_csv("../data/processed/Dimension/NCM.csv")

# Destination table
table_id = "foreign-trade-brazil.Dimension.NCM"

# Configurations
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Uploaded to BigQuery
job = client.load_table_from_dataframe(
    df_ncm,
    table_id,
    job_config=job_config
)

job.result()

print(f"{len(df_ncm)} records sent to {table_id}.")

13703 records sent to foreign-trade-brazil.Dimension.NCM.


### Mode of Transportation Code Table

In [57]:
via = pd.read_csv("../data/processed/Dimension/VIA.csv", 
                       sep = ",", 
                       encoding="latin1")


In [58]:
via.head()

,CO_VIA,NO_VIA,CO_VIA_EN
0,10,ENTRADA/SAIDA FICTA,Fictitious Entry/Exit
1,99,VIA DESCONHECIDA,Unknown
2,13,POR REBOQUE,By Tow
3,11,COURIER,Courier
4,15,VICINAL FRONTEIRICO,Border Local Road


In [59]:
# Connection with BigQuery
client = bigquery.Client(project="foreign-trade-brazil")

# Read the processed file
df_via = pd.read_csv("../data/processed/Dimension/VIA.csv")

# Destination table
table_id = "foreign-trade-brazil.Dimension.VIA"

# Configurations
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Uploaded to BigQuery
job = client.load_table_from_dataframe(
    df_via,
    table_id,
    job_config=job_config
)

job.result()

print(f"{len(df_via)} records sent to {table_id}.")

17 records sent to foreign-trade-brazil.Dimension.VIA.


In [60]:
via.columns

Index(['CO_VIA', 'NO_VIA', 'CO_VIA_EN'], dtype='object')

In [61]:
# Connection with BigQuery
client = bigquery.Client(project="foreign-trade-brazil")

# Read the processed file
df_urf = pd.read_csv("../data/processed/Dimension/URF.csv")

# Destination table
table_id = "foreign-trade-brazil.Dimension.URF"

# Configurations
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Uploaded to BigQuery
job = client.load_table_from_dataframe(
    df_urf,
    table_id,
    job_config=job_config
)

job.result()

print(f"{len(df_urf)} records sent to {table_id}.")

280 records sent to foreign-trade-brazil.Dimension.URF.
